# Tool Creation & Function Calling

In [ ]:
from langchain.tools import tool

### Calculator Tool

In [34]:
@tool
def calculator(operator: str, number1: float, number2: float):
    """
    Evaluate basic math operation
    Parameters:
    operator: +, -, * or /
    number1: first number in expression
    number2: second number in expression
    """
    if operator == "+":
        return str(number1 + number2)
    elif operator == "-":
        return str(number1 - number2)
    elif operator == "*":
        return str(number1 * number2)
    elif operator == "/":
        if number2 == 0:
            return "Cannot divide by 0"
        return str(number1 / number2)
    else:
        return "Invalid operator"
    

In [36]:
calculator.invoke({"operator": "*", "number1": 3, "number2": 4})

'12.0'

In [37]:
calculator.invoke({"operator": "/", "number1": 3, "number2": 4})

'0.75'

In [38]:
calculator.invoke({"operator": "+", "number1": 3, "number2": 4})

'7.0'

In [39]:
calculator.invoke({"operator": "-", "number1": 3, "number2": 4})

'-1.0'

### Web Search Tool

In [128]:
from ddgs import DDGS

@tool
def web_search(query: str):
    """
    Search the internet.
    Parameter:
    query: what you want to search
    """
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query,max_results=5))
            '''title = []
            url = []
            summary = []
            for res in results:
                title.append(res["title"])
                url.append(res["href"])
                summary.append(res["body"])   
            results = []'''
            formatted = []
            for i, res in enumerate(results):
                formatted.append(
                    f"{i+1}. {res['title']}\n"
                    f"Summary: {res['body']}\n"
                    f"URL: {res['href']}\n"
                )
            return "\n\n".join(formatted)

    except Exception as e:
        return str(e)

In [130]:
print(web_search.invoke({"query":"largest mammal"}))

1. Largest mammals
Summary: The following is a list of largest mammals by family.
URL: https://en.wikipedia.org/wiki/Largest_mammals


2. List of largest mammals - Wikipedia
Summary: The largest of this order and family of prickly-skinned, small mammals is the greater moonrat (Echinosorex gymnura), native to the rainforests of the Malaysian Peninsula as well as...
URL: https://en.wikipedia.org/wiki/List_of_largest_mammals


3. The Tapir Scientist: Saving South America's Largest Mammal (book)
Summary: The Tapir Scientist: Saving South America's Largest Mammal is a 2013 nonfiction children's book written by Sy Montgomery with photographs by Nic Bishop, part of the Scientists in the Field series published by Clarion Books. The book chronicles Montg…
URL: https://grokipedia.com/page/the_tapir_scientist_saving_south_americas_largest_mammal_(book)


4. Largest mammal | Guinness World Records
Summary: The largest mammal (and indeed the largest animal) on Earth based on weight is the blue whal

### File Reader Tool

In [28]:
@tool
def read_files(filename:str):
    """
    Read texts files 
    Parameter:
    filename: name of file (has to be a string)
    """
    try:
        with open(filename, "r") as file:
            return file.read()
    except Exception as e:
        return str(e)

In [ ]:
read_files.invoke({"filename":"text.txt"})

'lalala blablabla '

In [30]:
read_files.invoke({"filename":"not_real_file.txt"})

"[Errno 2] No such file or directory: 'not_real_file.txt'"

### Weather Tool

In [93]:
import requests

@tool
def weather(city: str) :
    """
    Checks the weather of a specific city
    Parameter:
    city: name of city that you want to check the weather of
    """
    try:
        url = "https://geocoding-api.open-meteo.com/v1/search"
        response = requests.get(url, params={"name": city.strip(), "count": 1, "format": "json"}, timeout=5)
        data = response.json()
        location = data["results"][0]
        latitude = location["latitude"]
        longitude = location["longitude"]
        weather_url = "https://api.open-meteo.com/v1/forecast"
        weather_response = requests.get(weather_url, params={"latitude": latitude, "longitude": longitude, "current_weather": True}, timeout=5)
        weather_data = weather_response.json()
        current = weather_data.get("current_weather", {})

        return (
            f"CURRENT WEATHER: {city.strip().upper()}\n"
            f"Temperature: {current.get('temperature', 'N/A')}°C\n"
            f"Wind Speed:  {current.get('windspeed', 'N/A')} km/h\n"
        )
        
    except Exception as e:
        return str(e)

In [ ]:
print(weather.invoke({"city":"singapore"}))

CURRENT WEATHER: SINGAPORE
Temperature: 29.0°C
Wind Speed:  10.1 km/h



In [94]:
print(weather.invoke({"city":"kuala lumpur"}))

CURRENT WEATHER: KUALA LUMPUR
Temperature: 28.6°C
Wind Speed:  3.1 km/h



In [ ]:
print(weather.invoke({"city":"kannur"}))

CURRENT WEATHER: KANNUR
Temperature: 26.3°C
Wind Speed:  6.6 km/h



### Email Sender Tool

Simulated

In [51]:
@tool
def email(receiver: str, subject: str, message: str):
    """
    Simulate sending an email
    Parameters:
    receiver: email address of receiver
    subject: subject of email
    message: message body 
    """
    valid = validate_email(receiver)
    if not valid:
        return "Please enter a valid email address"
    else:
        print("Email has been sent\n")
        formatted_email = (
            f"To:      {receiver}\n"
            f"Subject: {subject}\n"
            f"--------------------------------------------------\n"
            f"{message.strip()}\n"
            f"=================================================="
        )
        return formatted_email


import re
def validate_email(email: str):
    valid = bool(re.match(r"[a-zA-Z0-9_.]+@[a-zA-Z]+\.[a-zA-Z.]+",email))
    return valid


In [52]:
message = """
Dear Lecturer David,

I hope this email finds you well. Unfortunately, I have been unwell for the past few days and am still recovering. As a result, I am facing some challenges in completing the assigned task on time.
After assessing my current situation, I am writing to request a deadline extension of three additional days to submit the work. This extra time will allow me to complete the task to the best of my abilities and meet the expected standards.

I apologize for any inconvenience this may cause and appreciate your understanding in this matter. I am confident that with this brief extension, I will be able to submit quality work that meets your expectations.
Please let me know if this is feasible, and I will ensure that the work is submitted as soon as possible. If you require any further information or documentation, please do not hesitate to ask.
Thank you for your consideration, and I look forward to your response.

Best regards,
Sana
"""
print(email.invoke({"receiver":"sana1710@gmail.com", "subject":"Request for Deadline Extension due to Illness","message":message}))

Email has been sent

To:      sana1710@gmail.com
Subject: Request for Deadline Extension due to Illness
--------------------------------------------------
Dear Lecturer David,

I hope this email finds you well. Unfortunately, I have been unwell for the past few days and am still recovering. As a result, I am facing some challenges in completing the assigned task on time.
After assessing my current situation, I am writing to request a deadline extension of three additional days to submit the work. This extra time will allow me to complete the task to the best of my abilities and meet the expected standards.

I apologize for any inconvenience this may cause and appreciate your understanding in this matter. I am confident that with this brief extension, I will be able to submit quality work that meets your expectations.
Please let me know if this is feasible, and I will ensure that the work is submitted as soon as possible. If you require any further information or documentation, please d

### Date/Time Tool

In [23]:
from datetime import datetime, timedelta
@tool
def date_time(days):
    """ Get the current date/time or calculate a future date
    Parameters:
    days: 
    Calculate date after this many days 
    If 0, then return current date and time
    """
    date = datetime.now()
    if days == 0:
        return (f"It is currently {date.strftime('%c')}")
    elif days != 0:
         future_date = date + timedelta(days=days)
         return(f"The date after {days} days is {future_date.strftime('%d-%m-%Y')}")

In [ ]:
date_time.invoke({"days": 0})

'It is currently Sat Jun 27 15:47:49 2026'

In [25]:
date_time.invoke({"days": 30})

'The date after 30 days is 27-07-2026'

### Data Analyser Tool

In [ ]:
import pandas as pd

@tool
def data_analyser(filename:str, action: str):
    """ 
    Analyses csv data using pandas
    Parameters:
    filename: name of file
    action: 
    summary
    columns: columns of file
    entries: number of rows in file
    null: number of missing values in file
    sample: view first 5 entries in file
    """
    try:
        df = pd.read_csv(filename)
        if action.lower() == "summary":
            return df.describe()
        elif action.lower() == "columns":
            return f"Columns: {list(df.columns)}"
        elif action.lower() == "entries":
            return f"{df.shape[0]} entries"
        elif action.lower() == "null":
            return f"{df.isnull().sum().to_string()} missing values"
        elif action.lower() == "sample":
            return df.head()
        return "Supported actions are 'summary', 'columns', 'null', 'entries' and 'sample'"
    except Exception as e:
        return str(e)

In [60]:
data_analyser.invoke({"filename": "Titanic-Dataset.csv", "action": "summary"})

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [61]:
data_analyser.invoke({"filename": "Titanic-Dataset.csv", "action": "columns"})

"Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']"

In [62]:
data_analyser.invoke({"filename": "Titanic-Dataset.csv", "action": "entries"})

'891 entries'

In [68]:
print(data_analyser.invoke({"filename": "Titanic-Dataset.csv", "action": "null"}))

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2 missing values


In [69]:
data_analyser.invoke({"filename": "Titanic-Dataset.csv", "action": "sample"})

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Schema

In [132]:
tools = [
    {
        "name": "calculator",
        "description": "Do basic math operations: +, -, *, /",
        "parameters": {
            "type": "object",
            "properties": {
                "operator": {"type": "string", "description": "One of: +, -, *, /"},
                "number1":  {"type": "number"},
                "number2":  {"type": "number"}
            },
            "required": ["operator", "number1", "number2"]
        }
    },
    {
        "name": "weather",
        "description": "Get the current weather for a city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "Name of the city"}
            },
            "required": ["city"]
        }
    },
    {
        "name": "web_search",
        "description": "Search the internet",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"}
            },
            "required": ["query"]
        }
    },
    {
        "name": "read_files",
        "description": "Read and return the contents of a text file",
        "parameters": {
            "type": "object",
            "properties": {
                "filename": {
                    "type": "string",
                    "description": "Name of the file to read"
                }
            },
            "required": ["filename"]
        }
    },
    {
        "name": "email",
        "description": "Send an email to a recipient with a subject and message body",
        "parameters": {
            "type": "object",
            "properties": {
                "receiver": {
                    "type": "string",
                    "description": "Recipient's email address"
                },
                "subject": {
                    "type": "string",
                    "description": "Subject line of the email"
                },
                "message": {
                    "type": "string",
                    "description": "The content of the email"
                }
            },
            "required": ["receiver", "subject", "message"]
        }
    },
    {
        "name": "date_time",
        "description": "Get the current date/time, or calculate what the date will be after a number of days",
        "parameters": {
            "type": "object",
            "properties": {
                "days": {
                    "type": "integer",
                    "description": "Number of days to add to today. Use 0 for the current date and time."
                }
            },
            "required": ["days"]
        }
    },
    {
        "name": "data_analyser",
        "description": "Analyse a CSV file using pandas. Actions: summary, columns, entries, null, sample",
        "parameters": {
            "type": "object",
            "properties": {
                "filename": {
                    "type": "string",
                    "description": "Name of the CSV file, e.g. 'Titanic-Dataset.csv'"
                },
                "action": {
                    "type": "string",
                    "enum": ["summary", "columns", "entries", "null", "sample"],
                    "description": "summary=stats, columns=column names, entries=row count, null=missing values, sample=first 5 rows"
                }
            },
            "required": ["filename", "action"]
        }
    }
]